## The first big project - Professionally You!

### And, Tool use.

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import re
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
DEFAULT_MODEL = os.getenv("MODEL", "openai/gpt-4o-mini")
EVAL_MODEL = os.getenv("EVAL_MODEL", DEFAULT_MODEL)

if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY (or OPENAI_API_KEY) in your .env")

if "openrouter.ai" not in OPENROUTER_BASE_URL:
    print(f"Warning: OPENROUTER_BASE_URL was {OPENROUTER_BASE_URL}; overriding to OpenRouter.")
    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

_default_headers = {}
if os.getenv("OPENROUTER_SITE_URL"):
    _default_headers["HTTP-Referer"] = os.getenv("OPENROUTER_SITE_URL")
if os.getenv("OPENROUTER_APP_NAME"):
    _default_headers["X-Title"] = os.getenv("OPENROUTER_APP_NAME")

openai = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
    default_headers=_default_headers if _default_headers else None,
)

print("Using OpenRouter base_url:", openai.base_url)


Using OpenRouter base_url: https://openrouter.ai/api/v1/


In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [9]:
def push(message):
    if not pushover_user or not pushover_token:
        print("Pushover not configured; skipping push.")
        return
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    try:
        requests.post(pushover_url, data=payload, timeout=10)
    except Exception as e:
        print(f"Pushover failed: {e}")


In [10]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [11]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
FAQ = {
    "stack": "TypeScript, Python, LLM orchestration, agentic workflows, CI/CD, and DevOps.",
    "experience": "7+ years building full-stack systems and DevOps automation, with recent focus on AI-assisted engineering.",
    "llm": "Hands-on with Claude Code, Cursor, and Codex for code generation, benchmarking, and agent workflows.",
    "devops": "Built scalable CI/CD pipelines and automation for testing and delivery.",
    "testing": "Automated testing frameworks with Jest and Cypress to validate reliability of model-generated code.",
}

def lookup_faq(topic):
    if not topic:
        return {"answer": "No topic provided."}
    key = topic.strip().lower()
    for k, v in FAQ.items():
        if k in key:
            return {"answer": v}
    return {"answer": "No matching FAQ entry. Try: " + ", ".join(sorted(FAQ.keys()))}


def search_profile(query, max_results=5):
    if not query or not query.strip():
        return {"results": []}
    terms = [t for t in re.findall(r"\w+", query.lower()) if len(t) > 2]
    if not terms:
        return {"results": []}
    corpus = summary + "\n" + linkedin
    lines = [l.strip() for l in corpus.splitlines() if l.strip()]
    scored = []
    for line in lines:
        line_l = line.lower()
        score = sum(1 for t in terms if t in line_l)
        if score:
            scored.append((score, line))
    scored.sort(key=lambda x: x[0], reverse=True)
    return {"results": [line for _, line in scored[:max_results]]}


In [13]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [14]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [15]:
search_profile_json = {
    "name": "search_profile",
    "description": "Search the profile (summary + LinkedIn) and return the most relevant lines.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "What to search for in the profile"
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of lines to return"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

lookup_faq_json = {
    "name": "lookup_faq",
    "description": "Look up a short answer in a curated FAQ of common questions.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "Topic or keyword (e.g., stack, devops, testing, llm)"
            }
        },
        "required": ["topic"],
        "additionalProperties": False
    }
}


In [16]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json},
        {"type": "function", "function": search_profile_json},
        {"type": "function", "function": lookup_faq_json}]


In [17]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)


        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [19]:
globals()["record_unknown_question"]("this is a really hard question")

{'recorded': 'ok'}

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [21]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Michael Onyekanma"

In [22]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. You can use tools like search_profile and lookup_faq when a user asks for specific details or quick summaries. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [23]:
def evaluate_response(user_message, draft_response):
    evaluator_system = (
        "You are a strict evaluator for a professional personal-site assistant. "
        "Check for accuracy against the provided profile context, completeness, clarity, and professionalism. "
        "If improvements are needed, return a revised answer."
    )
    evaluator_user = (
        f"Summary:\n{summary}\n\n"
        f"LinkedIn/Profile:\n{linkedin}\n\n"
        f"User question:\n{user_message}\n\n"
        f"Draft answer:\n{draft_response}\n\n"
        "Return JSON with keys: verdict (pass|revise), feedback, improved_response."
    )

    try:
        eval_resp = openai.chat.completions.create(
            model=EVAL_MODEL,
            messages=[
                {"role": "system", "content": evaluator_system},
                {"role": "user", "content": evaluator_user},
            ],
        )
        content = eval_resp.choices[0].message.content or ""
        data = json.loads(content)
        verdict = str(data.get("verdict", "pass")).lower().strip()
        improved = (data.get("improved_response") or "").strip()
        if verdict == "revise" and improved:
            return improved
    except Exception:
        pass

    return draft_response


In [ ]:
def chat(user_message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": user_message}]
    done = False
    final_response = ""
    while not done:

        response = openai.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            done = True
            draft = response.choices[0].message.content
            final_response = evaluate_response(user_message, draft)
    return final_response


In [25]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
